[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Tensors


## What you will be able to do

Make tensors in PyTorch and in TensorFlow, say which dtype and device each one has, and move data
between them and NumPy, knowing which conversions copy the data and which share it.

## The idea

### The problem

Data rarely starts as a tensor. The stations in this guide arrive as a NumPy array of 1,200 rows
and four columns, and other data comes as Python lists, as columns of a table or as numbers read
from a file. A network wants tensors, and turning an array into one looks like a single call.

That call decides the tensor's dtype, which is easy to miss. NumPy holds decimals as 64-bit floats
and a network's weights are 32-bit, so an array passed straight in can fail at the first matrix
product, as the planned station's features did in the **Why PyTorch and TensorFlow** notebook. The
call also decides whether the tensor is a copy of the array or the same memory under a second name,
and with shared memory a change to the array changes the tensor too, with no error. Whether the
tensor can be changed afterward depends on the framework, since PyTorch allows it and TensorFlow
does not.

### A formal definition

> A **tensor** is an array of numbers of a single **dtype**, such as 32-bit floats or 64-bit
> integers, with a **shape** that gives its length along each dimension, held on one **device**, a
> CPU or a GPU. A tensor with no dimensions holds one number, a tensor with one dimension is a
> vector, and a tensor with two is a matrix.

### Why it works that way

Tensors follow NumPy's arrays closely, so most of what the **NumPy, Deep Dive** guide taught carries
over, and the differences are in the defaults and the rules.

For decimals written in Python, both frameworks choose 32-bit floats, because that is what their
layers compute in. For whole numbers, PyTorch chooses 64-bit integers, as NumPy does, and TensorFlow
32-bit ones. An array that comes from NumPy keeps NumPy's dtype in both, so a 64-bit array becomes a
64-bit tensor unless you ask for another.

When two dtypes meet, PyTorch converts both to one dtype that can hold the result. Two float tensors
give the wider float, as in NumPy, but whole numbers mixed with a decimal give 32-bit floats, where
NumPy gives 64-bit ones. TensorFlow raises an error instead, and one side has to be converted with
`tf.cast` first. A PyTorch tensor can be changed in place, while a TensorFlow tensor never changes once made, which is why TensorFlow has a second kind,
the `tf.Variable`, for values that training updates.

`torch.tensor` and `tf.constant` copy an array, while `torch.from_numpy` uses the array's own memory,
so it copies nothing however large the array is, and the two stay linked.

### Where this shows up

This guide's Setup makes the stations as 32-bit floats for these reasons. The **Automatic
Differentiation** notebook takes up `requires_grad`, which this notebook's Common errors meet once,
and the **Batches and Data Pipelines** notebook turns arrays into batches.

### What this notebook covers

- tensors from lists, the dtypes each framework chooses, and their shapes
- tensors of zeros, of ranges and of seeded random numbers
- tensors from NumPy arrays, and conversion between dtypes
- what each framework does when two dtypes meet
- the device a tensor is on
- passing data back to NumPy, and a tensor you can change
- four errors about dtypes, changing a tensor and shared memory

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import torch
import tensorflow as tf

values = np.array([1.5, 2.5, 3.5])
print(values.dtype)
print(torch.tensor(values).dtype, torch.tensor([1.5, 2.5, 3.5]).dtype)
print(tf.constant(values).dtype, tf.constant([1.5, 2.5, 3.5]).dtype)
```

```
float64
torch.float64 torch.float32
<dtype: 'float64'> <dtype: 'float32'>
```

The same three numbers become 64-bit tensors when they come from a NumPy array and 32-bit tensors
when they come from a list. Neither framework changes the dtype of an array it is given.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations and the arrays that tensors are made from here
- `torch` is PyTorch
- `tensorflow` is TensorFlow
- `contextlib` and `os` hide the messages TensorFlow prints while it loads

TensorFlow's start-up messages carry the time and a process number, so Setup hides them with
`TF_CPP_MIN_LOG_LEVEL` and `c_messages_hidden`, as in the **Why PyTorch and TensorFlow** notebook.

`station_network_arrays` makes the same made-up weather stations as that notebook, drawn from a
fixed seed so they are the same on every run. `features` has 1,200 rows, one per station, with its
latitude, longitude, elevation in meters and distance from the coast in kilometers, and `jan_mean`
holds each station's January mean in degrees Celsius. Both are 32-bit floats. The cell also splits
the stations and standardizes the features into `scaled`, which later notebooks use, and which this
notebook uses in one task.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


## Worked examples

### Tensors from lists

`torch.tensor` and `tf.constant` make a tensor from a Python list.

In [2]:
print(torch.tensor([1, 2, 3]))
print(torch.tensor([1.0, 2.0, 3.0]))
print(tf.constant([1, 2, 3]))
print(tf.constant([1.0, 2.0, 3.0]))

tensor([1, 2, 3])
tensor([1., 2., 3.])
tf.Tensor([1 2 3], shape=(3,), dtype=int32)
tf.Tensor([1. 2. 3.], shape=(3,), dtype=float32)


A PyTorch tensor prints its values, and names its dtype only when the dtype is not the default,
`int64` for whole numbers and `float32` for decimals, as the 64-bit tensor in the NumPy examples
shows. A TensorFlow tensor always prints its shape and dtype. The `.dtype` of each says which it chose, beside NumPy's choice for the same lists:

In [3]:
print("PyTorch:   ", torch.tensor([1, 2, 3]).dtype, torch.tensor([1.0, 2.0, 3.0]).dtype)
print("TensorFlow:", tf.constant([1, 2, 3]).dtype, tf.constant([1.0, 2.0, 3.0]).dtype)
print("NumPy:     ", np.array([1, 2, 3]).dtype, np.array([1.0, 2.0, 3.0]).dtype)

PyTorch:    torch.int64 torch.float32
TensorFlow: <dtype: 'int32'> <dtype: 'float32'>
NumPy:      int64 float64


Decimals become 32-bit floats in both frameworks, where NumPy makes 64-bit ones. Whole numbers become
64-bit integers in PyTorch, as in NumPy, and 32-bit integers in TensorFlow. TensorFlow prints a
dtype as `<dtype: 'int32'>`, and its name is the part in quotes.

### Shape, dimensions and size

The stations' features as a tensor in each framework:

In [4]:
table = torch.from_numpy(features)
table_tf = tf.constant(features)

print(table.shape, table.ndim, table.numel())
print(table_tf.shape, table_tf.ndim, int(tf.size(table_tf)))
print(table[0])
print(table_tf[0])

torch.Size([1200, 4]) 2 4800
(1200, 4) 2 4800
tensor([ 70.9300,  29.0700, 358.0000,  17.0000])
tf.Tensor([ 70.93  29.07 358.    17.  ], shape=(4,), dtype=float32)


`torch.from_numpy` and `tf.constant` turn the array into a tensor of the same shape, 1,200 rows of
4 values, with two dimensions and 4,800 values in all. PyTorch's shape is a `torch.Size` and
TensorFlow's a `TensorShape`, which prints as a tuple. Indexing works as it does in NumPy, so
`table[0]` is the first station's row. PyTorch prints every value of a float tensor with the same
number of decimal places, four here, while TensorFlow prints each value as NumPy would.

In [5]:
one = torch.tensor(3.5)
print(one.shape, one.ndim, one.item())
print(tf.constant(3.5).shape, float(tf.constant(3.5)))

torch.Size([]) 0 3.5
() 3.5


A tensor of one number has an empty shape and no dimensions. `.item()` returns its number as a
Python float, and `float()` does the same for a TensorFlow tensor.

### Tensors of zeros, ranges and evenly spaced values

In [6]:
print(torch.zeros(2, 3))
print(tf.zeros((2, 3)))
print(torch.arange(5), torch.arange(0, 1, 0.25))
print(tf.range(5))
print(torch.linspace(0, 1, 5))
print(tf.linspace(0.0, 1.0, 5))

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tf.Tensor(
[[0. 0. 0.]
 [0. 0. 0.]], shape=(2, 3), dtype=float32)
tensor([0, 1, 2, 3, 4]) tensor([0.0000, 0.2500, 0.5000, 0.7500])
tf.Tensor([0 1 2 3 4], shape=(5,), dtype=int32)
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
tf.Tensor([0.   0.25 0.5  0.75 1.  ], shape=(5,), dtype=float32)


PyTorch accepts a shape as separate numbers, `torch.zeros(2, 3)`, where TensorFlow's functions take
it as one tuple, `tf.zeros((2, 3))`. `arange` and `range` stop before their end, as Python's
`range` does, and `linspace` includes both ends. A range with whole-number steps is made of integers,
and one with a decimal step is made of floats.

### Random tensors from a seed

A network starts from random weights, so random tensors matter from the first layer on. A seed fixes
the numbers, which is how every notebook in this guide prints the same output on every run.

In [7]:
torch.manual_seed(0)
print(torch.randn(3))
torch.manual_seed(0)
print(torch.randn(3))

print(tf.random.Generator.from_seed(0).normal((3,)))
print(tf.random.Generator.from_seed(0).normal((3,)))

tensor([ 1.5410, -0.2934, -2.1788])
tensor([ 1.5410, -0.2934, -2.1788])
tf.Tensor([-1.3544159   0.70454913  0.03666191], shape=(3,), dtype=float32)
tf.Tensor([-1.3544159   0.70454913  0.03666191], shape=(3,), dtype=float32)


The same seed gives the same numbers within each framework. The two frameworks draw in their own
ways, so the seed 0 gives different numbers in each. `torch.manual_seed` resets PyTorch's draws
from that point on, and `tf.random.Generator.from_seed` makes a new TensorFlow generator from a seed.

### Tensors from NumPy arrays

A NumPy array keeps its dtype on the way in. Ask for another with `dtype=`, or convert the array
first with `astype`.

In [8]:
values = np.array([1.5, 2.5, 3.5])
print(torch.from_numpy(values))
print(values.dtype, torch.from_numpy(values).dtype, tf.constant(values).dtype)
print(torch.tensor(values, dtype=torch.float32).dtype, tf.constant(values, dtype=tf.float32).dtype)
print(torch.from_numpy(values.astype("float32")).dtype)

tensor([1.5000, 2.5000, 3.5000], dtype=torch.float64)
float64 torch.float64 <dtype: 'float64'>
torch.float32 <dtype: 'float32'>
torch.float32


The 64-bit array became a 64-bit tensor in both frameworks, and PyTorch names `torch.float64` when it
prints one. `dtype=torch.float32` and
`dtype=tf.float32` make a 32-bit tensor from it, and `astype("float32")` makes a 32-bit array first,
which `torch.from_numpy` then uses as it is.

### Converting between dtypes

`.to` converts a PyTorch tensor and `tf.cast` a TensorFlow one. `.float()` is PyTorch's short form
for `.to(torch.float32)`.

In [9]:
readings = torch.tensor([-1.7, 1.7, 2.5])
print(readings.to(torch.int64), readings.round().to(torch.int64))

readings_tf = tf.constant([-1.7, 1.7, 2.5])
print(tf.cast(readings_tf, tf.int32), tf.cast(tf.round(readings_tf), tf.int32))

print(torch.tensor([1, 2]).float(), tf.cast(tf.constant([1, 2]), tf.float32))

tensor([-1,  1,  2]) tensor([-2,  2,  2])
tf.Tensor([-1  1  2], shape=(3,), dtype=int32) tf.Tensor([-2  2  2], shape=(3,), dtype=int32)
tensor([1., 2.]) tf.Tensor([1. 2.], shape=(2,), dtype=float32)


Converting a float to a whole number drops the fraction, so −1.7 becomes −1 and 1.7 becomes 1, which
is toward zero in both directions. Round first to get the nearest whole number. Both frameworks
round 2.5 to 2, because a value exactly halfway goes to the even neighbor.

### When two dtypes meet

PyTorch converts a mix of dtypes to one dtype that can hold the result.

In [10]:
whole = torch.tensor([1, 2])
print(whole + 0.5, (whole + 0.5).dtype)
print(whole / 2, (whole / 2).dtype)
print((torch.ones(2) + torch.ones(2, dtype=torch.float64)).dtype)
print((np.array([1, 2]) + 0.5).dtype)

tensor([1.5000, 2.5000]) torch.float32
tensor([0.5000, 1.0000]) torch.float32
torch.float64
float64


A whole number plus a decimal gives 32-bit floats, and so does dividing whole numbers with `/`. A
32-bit tensor plus a 64-bit tensor gives a 64-bit tensor. NumPy, in the last line, makes the same
whole numbers plus 0.5 into 64-bit floats.

In [11]:
whole_tf = tf.constant([1, 2])
print(whole_tf / 2)
print(tf.cast(whole_tf, tf.float32) + 0.5)

tf.Tensor([0.5 1. ], shape=(2,), dtype=float64)
tf.Tensor([1.5 2.5], shape=(2,), dtype=float32)


TensorFlow divides whole numbers into 64-bit floats, where PyTorch gave 32-bit ones. To add 0.5 to
whole numbers, the cell casts them to floats first, because TensorFlow adds tensors only when their
dtypes match, as the first of the Common errors shows.

### The device a tensor is on

In [12]:
print(torch.ones(2).device, torch.cuda.is_available())
print(tf.ones(2).device)
print(tf.config.list_physical_devices())

cpu False
/job:localhost/replica:0/task:0/device:CPU:0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


Every tensor here is on the CPU, the only device this runtime has. On a runtime with a GPU, PyTorch
moves a tensor there with `.to("cuda")` and raises an error when an operation's inputs are on
different devices, while TensorFlow places its tensors on the GPU by itself and copies an input to
the device an operation runs on. This notebook runs on a CPU, so it cannot show what a
GPU prints.

### Back to NumPy

`.numpy()` turns a tensor of either framework into a NumPy array, and `.item()` or `float()` turns
a tensor of one value into a Python number. PyTorch's `.numpy()` returns an array that shares the
tensor's memory, so it shares the original array's memory too when the tensor came from
`torch.from_numpy`. TensorFlow's `.numpy()` always returns a copy. `np.shares_memory` says whether
two arrays share memory.

In [13]:
shared = torch.from_numpy(features)
copied = torch.tensor(features)
print(np.shares_memory(features, shared.numpy()), np.shares_memory(features, copied.numpy()))

back = copied.numpy()
back[0, 0] = 0.0
print(copied[0, 0].item())

from_tf = table_tf.numpy()
from_tf[0, 0] = 0.0
print(from_tf[0, 0], table_tf[0, 0].numpy())

True False
0.0
0.0 70.93


`torch.from_numpy` shares the array's memory and `torch.tensor` copies it, and the array
`copied.numpy()` returned shares `copied`'s memory, so setting its first value set the tensor's.
Changing the array that TensorFlow's `.numpy()` returned left the tensor at 70.93, because
TensorFlow's `.numpy()` always copies. Sharing saves copying a large array, and the last of the
Common errors shows its cost.

### A tensor you can change

A PyTorch tensor can be changed by assigning to part of it, or by a method whose name ends in an
underscore, which changes the tensor in place. In TensorFlow, a `tf.Variable` holds a value that can
change, through its `assign` methods.

In [14]:
row = torch.tensor([1.0, 2.0, 3.0])
row[0] = 10.0
row.add_(1.0)
print(row)

weights = tf.Variable([1.0, 2.0, 3.0])
weights[0].assign(10.0)
weights.assign_add([1.0, 1.0, 1.0])
print(weights)
print(weights + 1)

tensor([11.,  3.,  4.])
<tf.Variable 'Variable:0' shape=(3,) dtype=float32, numpy=array([11.,  3.,  4.], dtype=float32)>
tf.Tensor([12.  4.  5.], shape=(3,), dtype=float32)


`row` ends as 11, 3 and 4, and the variable went through the same steps. Arithmetic on a variable
gives an ordinary `tf.Tensor`, and only the `assign` methods change the variable.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/02-tensors-solutions.ipynb).

**1.** Make the whole numbers 1 to 10 as a PyTorch tensor with `torch.arange` and as a TensorFlow tensor with `tf.range`, and print each one's dtype.

In [15]:
# your code here

**2.** Make a TensorFlow tensor of the first five January means in `jan_mean` as 64-bit floats, cast it to 32-bit floats, and print both dtypes and the cast values.

In [16]:
# your code here

**3.** Draw four numbers from a normal distribution in PyTorch from the seed 42, twice, and check with `torch.equal` that the two draws match. Then do the same in TensorFlow with `tf.random.Generator.from_seed(42)` and `tf.reduce_all(a == b)`.

In [17]:
# your code here

**4.** Make a `tf.Variable` holding the first five January means, add 1.0 to the third of them with `assign`, and print the variable.

In [18]:
# your code here

**5.** Make `sample = scaled[:3].copy()`, then one tensor from it with `torch.from_numpy` and one with `torch.tensor`. Set `sample[0, 0] = 0.0` and print the first value of each tensor.

In [19]:
# your code here

## Common errors

### `InvalidArgumentError: cannot compute AddV2 as input #1(zero-based) was expected to be a float tensor but is a double tensor [Op:AddV2] name:`

A correction for three stations' January means was worked out in NumPy, and is added to their
values in TensorFlow.

In [20]:
corrections = np.array([0.5, -0.2, 0.1])
tf.constant(jan_mean[:3]) + tf.constant(corrections)

InvalidArgumentError: cannot compute AddV2 as input #1(zero-based) was expected to be a float tensor but is a double tensor [Op:AddV2] name: 

TensorFlow names the operation, `AddV2`, and the input it rejected, counting from zero, so `#1` is
the second operand of `+`, the corrections. `float` is TensorFlow's word here for 32-bit floats and
`double` for 64-bit ones. `jan_mean` is 32-bit, and `np.array` made the corrections 64-bit.
TensorFlow does not convert either one to match the other.

In [21]:
print((tf.constant(jan_mean[:3]) + tf.constant(corrections, dtype=tf.float32)).numpy().round(2))
print((torch.from_numpy(jan_mean[:3]) + torch.from_numpy(corrections)).dtype)

[-9.1 -3.9 -4.7]
torch.float64


Making the corrections 32-bit fixes it. PyTorch runs the same sum without an error and promotes the
result to 64-bit floats, which a 32-bit network then rejects, as the **Why PyTorch and TensorFlow**
notebook showed.

### `TypeError: 'tensorflow.python.framework.ops.EagerTensor' object does not support item assignment`

In [22]:
january = tf.constant(jan_mean[:5])
january[0] = -9.0

TypeError: 'tensorflow.python.framework.ops.EagerTensor' object does not support item assignment

An `EagerTensor` is the ordinary TensorFlow tensor, and it cannot be changed once made. A value that
needs changing belongs in a `tf.Variable`, or in the NumPy array before the tensor is made.

In [23]:
january = tf.Variable(jan_mean[:5])
january[0].assign(-9.0)
print(january.numpy())

[-9.  -3.7 -4.8 -9.4 -8.2]


The variable now holds −9.0 in its first place.

### `RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.`

In [24]:
w = torch.tensor([0.5, -1.0], requires_grad=True)
w.numpy()

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

A tensor made with `requires_grad=True` records what is computed from it, and a NumPy array has no
place for that record, so PyTorch will not hand the tensor over as one. `detach()` returns the same
values with no record attached, as the message says.

In [25]:
print(w.detach().numpy())

[ 0.5 -1. ]


### No error, and a tensor that changed when its array did

In [26]:
readings = np.array([1.5, 2.5, 3.5], dtype=np.float32)
before = torch.from_numpy(readings)
readings[0] = 99.0
print(before)

tensor([99.0000,  2.5000,  3.5000])


The tensor shows 99.0, a value set on the array after the tensor was made, because
`torch.from_numpy` shares the array's memory, and nothing warns that the tensor changed.

In [27]:
readings = np.array([1.5, 2.5, 3.5], dtype=np.float32)
before = torch.tensor(readings)
readings[0] = 99.0
print(before)

tensor([1.5000, 2.5000, 3.5000])


`torch.tensor` copies the array, so the tensor keeps the values it was made with.

## Recap

- You can make tensors from lists, ranges, seeds and NumPy arrays, and say which dtype each gets.
- You can convert a tensor's dtype with `.to` or `tf.cast`, and know that converting to whole
  numbers drops the fraction.
- You can predict what each framework does when two dtypes meet in a sum.
- You can change a PyTorch tensor in place, and a TensorFlow value through a `tf.Variable`.
- You can tell a conversion that copies from one that shares memory.

## What is next

The **Shapes and Broadcasting** notebook changes a tensor's shape without changing its values, and
lines up tensors of different shapes in one sum.

---

&#8592; **Previous:** [Why PyTorch and TensorFlow](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/01-why-pytorch-and-tensorflow.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Shapes and Broadcasting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/03-shapes-and-broadcasting.ipynb) &#8594;
